In [ ]:
import numpy as np
import rasterio as rio
from rasterio.warp import reproject, Resampling
from rasterio.enums import Resampling
from pyproj import CRS
from matplotlib import pyplot as plt
from pathlib import Path
from tqdm import tqdm
import os

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [ ]:
GDP_2020_ = rio.open(DATA / 'GDP-2010-2020' / 'GDP2020.tif')

In [15]:
GDP_2020 = GDP_2020_.read(1)

In [20]:
print(np.max(GDP_2020), np.min(GDP_2020))
print(np.sum(GDP_2020))

2.6765087e+10 0.0
1.0936774e+14


In [21]:
GDP_2020_1km = np.zeros(shape=country_raster_1km_.shape, dtype=np.float32)

In [22]:
reproject(
    source=GDP_2020,
    destination=GDP_2020_1km,
    src_transform=GDP_2020_.transform,
    src_crs=GDP_2020_.crs,
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.sum,
    src_nodata=None,
    dst_nodata=0
)

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(16922, 33844), dtype=float32),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [23]:
print(np.max(GDP_2020_1km), np.min(GDP_2020_1km))
print(np.sum(GDP_2020_1km))

2.8112214e+10 0.0
1.0936686e+14


In [24]:
print((np.sum(GDP_2020_1km) - np.sum(GDP_2020)) / np.sum(GDP_2020))

-8.053598e-06


In [ ]:
with rio.open(
    RESULTS / 'GDP-rasters' / 'GDP-2020-1km.tif',
    "w",
    driver="GTiff",
    height=country_raster_1km_.shape[0],
    width=country_raster_1km_.shape[1],
    count=1,
    dtype=GDP_2020_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(GDP_2020_1km, 1)

In [ ]:
def reproject_GDP_data(year, SSP):
    input_path = DATA / 'future-gdp-1km' / f'SSP{SSP}' / f'GDP{year}_ssp{SSP}.tif'
    src = rio.open(input_path)
    src_data = src.read(1)
    dst_data = np.zeros(shape=country_raster_1km_.shape, dtype=np.float32)
    reproject(
        source=src_data,
        destination=dst_data,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=country_raster_1km_.transform,
        dst_crs=country_raster_1km_.crs,
        resampling=Resampling.sum,
        src_nodata=None,
        dst_nodata=0
    )

    print((np.sum(dst_data) - np.sum(src_data)) / np.sum(src_data))

    return dst_data

In [ ]:
def output_raster(data, year, SSP):
    with rio.open(
        RESULTS / 'GDP-rasters' / f'GDP-{year}-SSP{SSP}-1km.tif',
        "w",
        driver="GTiff",
        height=country_raster_1km_.shape[0],
        width=country_raster_1km_.shape[1],
        count=1,
        dtype=data.dtype,
        crs=country_raster_1km_.crs,
        transform=country_raster_1km_.transform,
        compress='lzw'
    ) as dst:
        dst.write(data, 1)

In [ ]:
for SSP in range(1, 6):
    # print(SSP)
    for t in tqdm(range(2025, 2055, 5)):
        # print(t)
        result = reproject_GDP_data(t, SSP)
        output_raster(result, t, SSP)